# Prometheus v1 — Colab Runtime Template

**Version / Updated:** `2026-11-24 17:02 PST` *(Phase 2 `v0.3.0`: Hofstadter Analogy Transfer + Ashby Ultrastability Monitor + Bounded Self-Improvement Loop Cell 4e)*

**Do not edit protected cells manually.** The harness layer may be updated by
the governor-approved Ashby loop; the gate and checkpoint cells are protected.

## How a session works
1. **Cell 1 – Config**: set `RUN_ID`, `AUDIT_SEEDS`, and `MODEL_PINS`.
2. **Cell 2 – Mount & install**: mounts Drive, installs the newest `prometheus-v1`
   wheel (`0.3.0`) from Drive, initialises the persistent Gate.
3. **Cell 3 – Gate check**: verifies the governor heartbeat; aborts if halted.
4. **Cell 4 – Phase 2 Cognitive Run Loop**: runs `CognitiveAgent` with Friston EFE,
   Good's MDL `AssemblyMemory`, **Hofstadter `StructuralAnalogyTransfer`**, and
   **Ashby `HomeostaticMonitor`** (plus optional live Qwen3-8B hybrid advisor),
   persisting `memory_state.json`, `ashby_state.json`, `phase2_checkpoint.json`,
   and hash-chained `logs/agent.jsonl` to Drive after every episode.
5. **Cell 4b – vLLM Launcher & Foundation-Model Bake-off**: auto-detects your Colab
   GPU (L4 / A100 / G4 / T4), launches vLLM (with pinned `qwen3-8b` commit
   `b968826d9c46dd6066d109eabc6255188de91218`), and verifies `/v1/models`.
6. **Cell 4c – Audit-seed evaluation**: runs `CognitiveAgent` (+ transfer) in read-only
   memory mode (`learn=False`) across `AUDIT_SEEDS` (`1000–1049`).
7. **Cell 4d – Lean 4 prover bake-off**: installs `elan` + Lean 4 and benchmarks
   prover candidates (`goedel-prover-v2-8b`, etc.) via `run_prover_episode`.
8. **Cell 4e – Phase 2 Bounded Ashby Self-Improvement Loop**: screens homeostatic
   harness mutations on held-out dev seeds, audits the winner read-only on `AUDIT_SEEDS`,
   and publishes a signed-approval proposal (`g3`) to Drive for the workstation governor.
9. **Cell 5 / 6 – Manual propose & Wait/Install variant (`g3`)**: verifies the governor's
   signed approval and installs onto local ext4 (`INSTALL_ROOT`).

After a Colab forced restart, re-run from **Cell 1 → Cell 2** (gate state,
`memory_state.json`, `ashby_state.json`, and checkpoints resume automatically from Drive).

---
## Cell 1 — Configuration *(edit this cell)*

In [ ]:
# ── User-editable configuration ───────────────────────────────────────────────
RUN_ID = "run-001"          # must match the governor's policy run_id

# Must match the governor's live policy.json (audit_seeds and model_pins).
# Keep MODEL_PINS = {} while continuing run-001; when starting a fresh pinned run-002,
# match policy.json e.g. {"cortex": "qwen3-8b@b968826d9c46dd6066d109eabc6255188de91218"}.
AUDIT_SEEDS = list(range(1000, 1050))
MODEL_PINS  = {}

# Google Drive paths (relative to MyDrive root)
DRIVE_ROOT      = "/content/drive/MyDrive"
PROMETHEUS_DIR  = f"{DRIVE_ROOT}/prometheus"          # gdrive:prometheus
EXCHANGE_DIR    = f"{PROMETHEUS_DIR}/{RUN_ID}"         # gdrive:prometheus/run-001
PUB_KEY_PATH    = f"{PROMETHEUS_DIR}/governor_public_key.txt"
WHEEL_GLOB      = f"{PROMETHEUS_DIR}/prometheus_v1-*.whl"  # pre-built wheel

# INSTALL_ROOT MUST live on Colab's local ext4 disk (/content/...), NOT Drive FUSE,
# because Gate.install() uses atomic POSIX symlinks (`current -> gen-XXXX`) which
# Google Drive FUSE rejects with Errno 95 (Operation not supported).
INSTALL_ROOT          = "/content/installed_harness"

# Persistent state, memory, Ashby monitor, logs & checkpoints on Drive
GATE_STATE_PATH       = f"{EXCHANGE_DIR}/gate_state.json"
CHECKPOINT_PATH       = f"{EXCHANGE_DIR}/checkpoint.json"
PHASE1_CKPT_PATH      = f"{EXCHANGE_DIR}/phase1_checkpoint.json"
PHASE2_CKPT_PATH      = f"{EXCHANGE_DIR}/phase2_checkpoint.json"
AUDIT_CHECKPOINT_PATH = f"{EXCHANGE_DIR}/audit_checkpoint.json"
MEMORY_STATE_PATH     = f"{EXCHANGE_DIR}/memory_state.json"
ASHBY_STATE_PATH      = f"{EXCHANGE_DIR}/ashby_state.json"
AGENT_LOG_PATH        = f"{EXCHANGE_DIR}/logs/agent.jsonl"

# Heartbeat tolerance (should match governor policy: heartbeat_ttl_s = 900)
MAX_CLOCK_SKEW_S = 60.0

print(f"RUN_ID        : {RUN_ID}")
print(f"Exchange dir  : {EXCHANGE_DIR}")
print(f"Public key    : {PUB_KEY_PATH}")
print(f"Install root  : {INSTALL_ROOT}  (local ext4)")
print(f"Gate state    : {GATE_STATE_PATH}")
print(f"Phase 2 ckpt  : {PHASE2_CKPT_PATH}")
print(f"Memory state  : {MEMORY_STATE_PATH}")
print(f"Ashby state   : {ASHBY_STATE_PATH}")
print(f"Agent log     : {AGENT_LOG_PATH}")

---
## Cell 2 — Mount Drive & install package
*Re-run Cell 1 and Cell 2 after every forced restart.*

In [ ]:
import os, sys, json, time, glob, shutil, hashlib, subprocess, pathlib

# ── 1. Mount Google Drive ─────────────────────────────────────────────────────
from google.colab import drive
drive.mount("/content/drive", force_remount=False)
print("Drive mounted.")

# ── 2. Verify exchange directory and public key exist ─────────────────────────
for p in [EXCHANGE_DIR, PUB_KEY_PATH]:
    if not os.path.exists(p):
        raise FileNotFoundError(
            f"Expected path not found: {p}\n"
            "Check that the governor has started and rclone has synced."
        )
print(f"Exchange dir  : OK  ({EXCHANGE_DIR})")

GOV_PUB_KEY_HEX = pathlib.Path(PUB_KEY_PATH).read_text().strip()
assert len(GOV_PUB_KEY_HEX) == 64, "governor_public_key.txt should be a 64-char hex string"
print(f"Governor key  : {GOV_PUB_KEY_HEX[:16]}…  (loaded)")

# ── 3. Install prometheus-v1 from the newest wheel on Drive ───────────────────
wheels = sorted(glob.glob(WHEEL_GLOB))
if wheels:
    wheel = wheels[-1]   # newest by version/filename
    print(f"Installing from wheel: {os.path.basename(wheel)}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "cryptography>=42"])
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--quiet",
        "--force-reinstall", "--no-deps", wheel,
    ])
else:
    raise FileNotFoundError(
        f"No wheel matching {WHEEL_GLOB} found on Drive.\n"
        "Build one locally (`pip wheel --no-deps -w dist .`) and upload dist/*.whl to Drive."
    )

# ── 4. Initialise the persistent Gate ─────────────────────────────────────────
import prometheus
from prometheus.runtime.gate import Gate, Halted, InstallRefused

baseline_hash_path = pathlib.Path(EXCHANGE_DIR) / "baseline_hash.txt"
if not baseline_hash_path.exists():
    raise FileNotFoundError(
        f"{baseline_hash_path} not found.\n"
        "Run `prometheus-governor init` and sync baseline_hash.txt from the workstation first."
    )
BASELINE_HASH = baseline_hash_path.read_text().strip()
print(f"Package ver   : {prometheus.__version__}")
print(f"Baseline hash : {BASELINE_HASH[:16]}…")

gate = Gate(
    governor_public_key_hex=GOV_PUB_KEY_HEX,
    run_id=RUN_ID,
    exchange=pathlib.Path(EXCHANGE_DIR),
    install_root=pathlib.Path(INSTALL_ROOT),
    baseline_hash=BASELINE_HASH,
    max_clock_skew_s=MAX_CLOCK_SKEW_S,
    state_path=pathlib.Path(GATE_STATE_PATH),
)
print(f"Gate ready    : generation={gate.generation}, "
      f"installed={gate.installed_hash[:16]}…")
harness_active = (pathlib.Path(INSTALL_ROOT) / "current" / "harness").is_dir()
print(f"Local harness : {'active on /content' if harness_active else 'defaults (will populate after next Cell 6 install)'}")

---
## Cell 3 — Gate check
Run once after Cell 2. Cells 4, 4b, 4c, 4d, and 4e call this before every step
automatically; this cell verifies the live workstation governor heartbeat manually.

In [ ]:
try:
    hb = gate.check_heartbeat()
    remaining = hb["expires_at"] - time.time()
    print(f"Heartbeat OK  : counter={hb['counter']}, "
          f"expires in {remaining:.0f}s, halt={hb.get('halt', False)}")
except Halted as e:
    print(f"HALTED: {e}")
    raise

---
## Cell 4 — Phase 2 Cognitive Run Loop *(EFE + MDL Memory + Hofstadter Transfer + Ashby Monitor)*

Runs `CognitiveAgent` over fresh Hidden-Law Lab worlds (`seeds 200–299`, disjoint from
Phase 1 training `0–199` and frozen `AUDIT_SEEDS` `1000–1049`) with:
- **Hofstadter `StructuralAnalogyTransfer`** retrieving structural motifs from your
  24 consolidated Phase 1 assemblies in `memory_state.json`.
- **Ashby `HomeostaticMonitor`** watching immutable `EssentialBounds` and persisting
  `ashby_state.json` to Drive for the local Observatory.
- Optional **Hybrid Qwen3-8B Advisor** (`USE_HYBRID_LLM = True` after Cell 4b-setup).

In [ ]:
import dataclasses, json, pathlib
from prometheus.bakeoff import MODEL_REGISTRY
from prometheus.common.eventlog import EventLog
from prometheus.core import (
    AssemblyMemory,
    CognitiveAgent,
    HomeostaticMonitor,
    StructuralAnalogyTransfer,
    load_harness,
)
from prometheus.harness.client import ModelClient, ModelClientError
from prometheus.lab import HiddenLawWorld
from prometheus.lab.laws import LAW_NAMES
from prometheus.runtime.gate import Halted

# ── Phase 2 episode configuration ─────────────────────────────────────────────
EPISODE_SEEDS  = list(range(200, 300))   # 100 fresh worlds (disjoint from 0-199 & 1000-1049)
TOTAL_EPISODES = len(EPISODE_SEEDS)
NOISE          = 0.05
BUDGET         = 500.0                   # try 25.0 to stress-test tight-budget transfer + Ashby!
USE_TRANSFER   = True                    # Hofstadter structural analogy transfer
USE_HYBRID_LLM = False                   # set True after starting vLLM in Cell 4b-setup
HYBRID_MODEL   = globals().get("MODEL_NAME", "qwen3-8b")
VLLM_URL       = globals().get("VLLM_URL", "http://localhost:8000/v1")

# ── Load active harness, MDL memory, Hofstadter transfer, Ashby monitor, log ──
harness_dir = pathlib.Path(INSTALL_ROOT) / "current" / "harness"
harness = load_harness(harness_dir if harness_dir.is_dir() else None)

client = None
if USE_HYBRID_LLM:
    spec = MODEL_REGISTRY[HYBRID_MODEL]
    client = ModelClient(
        base_url=VLLM_URL,
        model_id=spec.hf_id,
        temperature=0.0,
        max_tokens=spec.default_max_tokens,
        extra_body=dict(spec.extra_body),
    )
    if not client.health_check():
        raise ModelClientError(
            f"USE_HYBRID_LLM=True but vLLM at {VLLM_URL} is not serving {spec.hf_id!r}. "
            "Run Cell 4b-setup first, or set USE_HYBRID_LLM = False."
        )
    harness = dataclasses.replace(
        harness, mesh=dataclasses.replace(harness.mesh, use_llm_advisor=True)
    )

mem_path = pathlib.Path(MEMORY_STATE_PATH)
ashby_path = pathlib.Path(ASHBY_STATE_PATH)
memory = AssemblyMemory.load(mem_path, params_override=harness.memory_params)
agent_log = EventLog(pathlib.Path(AGENT_LOG_PATH))
transfer = StructuralAnalogyTransfer(harness.transfer_params) if USE_TRANSFER else None
monitor = HomeostaticMonitor.load(
    ashby_path, config_override=harness.ashby_config, event_log=agent_log
)
agent = CognitiveAgent(
    harness=harness,
    memory=memory,
    client=client,
    transfer=transfer,
    monitor=monitor,
    event_log=agent_log,
    rng_seed=200,
)

print(f"Loaded harness : {harness.loaded_files or ('defaults',)}")
print(f"Transfer       : {'ON (Hofstadter structural analogy)' if transfer else 'OFF'}")
print(f"Hybrid LLM     : {'ON (' + HYBRID_MODEL + ')' if client else 'OFF'}")
print(f"Memory loaded  : {len(memory.assemblies)} assemblies, {memory.episodes_consolidated} consolidated")
print(f"Ashby monitor  : {monitor.status_summary()['status']} (triggers={monitor.trigger_count})")

# ── Checkpoint helpers ────────────────────────────────────────────────────────
ckpt_path = pathlib.Path(PHASE2_CKPT_PATH)

def load_phase2_checkpoint() -> dict:
    if ckpt_path.exists():
        st = json.loads(ckpt_path.read_text(encoding="utf-8"))
        print(f"Checkpoint resumed: episode {st['next_episode']}/{TOTAL_EPISODES}, "
              f"{len(st['results'])} results so far")
        return st
    print("No Phase 2 checkpoint found — starting fresh.")
    return {"next_episode": 0, "results": {}}

def save_phase2_checkpoint(st: dict) -> None:
    tmp = ckpt_path.with_name(f".{ckpt_path.name}.tmp")
    tmp.write_text(json.dumps(st, indent=2, sort_keys=True), encoding="utf-8")
    tmp.replace(ckpt_path)

state = load_phase2_checkpoint()

for episode_idx in range(state["next_episode"], TOTAL_EPISODES):
    seed = EPISODE_SEEDS[episode_idx]

    try:
        gate.check_heartbeat()
    except Halted as e:
        memory.save(mem_path)
        monitor.save(ashby_path)
        save_phase2_checkpoint(state)
        print(f"\nPhase 2 run halted by governor at episode {episode_idx}: {e}")
        raise SystemExit(0)

    world = HiddenLawWorld(seed=seed, noise=NOISE, budget=BUDGET)
    res = agent.run_episode(world, learn=True, episode_idx=200 + episode_idx)
    truth = world.ground_truth()
    correct = sum(res.claims.get(k) == v for k, v in truth.items())

    state["results"][str(seed)] = {
        "claims": res.claims,
        "truth": truth,
        "correct": correct,
        "family": world.family,
        "n": res.n,
        "spent": round(res.budget_spent, 3),
        "observations": res.observations_used,
        "entropy_bits": round(res.final_entropy_bits, 4),
        "mdl_gain_bits": round(res.metrics.mdl_compression_bits, 4),
        "stopped_early": res.stopped_early,
        "transfer_hints": res.transfer_hints,
        "transfer_source": res.transfer_source,
        "transfer_confidence": res.transfer_confidence,
        "ashby_triggered": res.ashby_triggered,
        "llm_calls": res.metrics.llm_calls,
        "latency_s": round(res.latency_s, 3),
    }
    state["next_episode"] = episode_idx + 1
    memory.save(mem_path)
    monitor.save(ashby_path)
    save_phase2_checkpoint(state)

    if (episode_idx + 1) % 10 == 0 or episode_idx == TOTAL_EPISODES - 1 or res.ashby_triggered:
        done = state["next_episode"]
        vals = list(state["results"].values())
        avg_acc = sum(r["correct"] for r in vals) / (done * len(LAW_NAMES))
        avg_spent = sum(r["spent"] for r in vals) / done
        hint_pct = sum(1 for r in vals if r.get("transfer_hints", 0) > 0) / done * 100.0
        st_lbl = monitor.status_summary()["status"]
        tag = "  [ASHBY TRIGGERED!]" if res.ashby_triggered else ""
        print(
            f"Episode {done:3d}/{TOTAL_EPISODES}  acc={avg_acc:.4f}  "
            f"budget={avg_spent:5.1f}/{BUDGET:.0f}  analogy={hint_pct:4.1f}%  "
            f"ashby={st_lbl}{tag}"
        )

vals = list(state["results"].values())
total = len(vals)
avg_acc = sum(r["correct"] for r in vals) / (total * len(LAW_NAMES))
avg_spent = sum(r["spent"] for r in vals) / total
hint_pct = sum(1 for r in vals if r.get("transfer_hints", 0) > 0) / total * 100.0
print()
print("═══ Phase 2 CognitiveAgent complete ═══")
print(f"Accuracy         : {avg_acc:.4f}")
print(f"Mean budget/ep   : {avg_spent:.2f} / {BUDGET:.0f}")
print(f"Analogy active   : {hint_pct:.1f}% of episodes")
if transfer:
    print(f"Transfer stats   : {transfer.telemetry()}")
print(f"Ashby status     : {monitor.status_summary()}")

---
## Cell 4b — vLLM Launcher & Foundation-Model Bake-off *(Phase 0 / Hybrid Cortex)*

Auto-detects your Colab GPU (**L4 / A100 / G4** run unquantized BF16 / native MXFP4;
**T4** installs `vllm-bnb-plugin` + `bitsandbytes` 4-bit), launches vLLM (with pinned
`qwen3-8b` commit `b968826d9c46dd6066d109eabc6255188de91218`), waits until `/v1/models`
is healthy, and serves `http://localhost:8000/v1`.

In [ ]:
# ── Cell 4b-setup: detect GPU, install vLLM, launch server & wait until ready ─
import importlib.util, json, os, pathlib, signal, subprocess, sys, time, urllib.request
import torch
from prometheus.bakeoff import MODEL_REGISTRY, vllm_flags_for_gpu

MODEL_NAME = "qwen3-8b"

if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU detected — switch Runtime → Change runtime type → GPU (L4/A100/G4/T4).")
gpu_name = torch.cuda.get_device_name(0)
compute_cap = torch.cuda.get_device_capability(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
print(f"GPU           : {gpu_name}  ({vram_gb:.1f} GB, SM {compute_cap[0]}.{compute_cap[1]})")

if not os.environ.get("HF_TOKEN"):
    try:
        from google.colab import userdata
        tok = userdata.get("HF_TOKEN")
        if tok:
            os.environ["HF_TOKEN"] = tok
            os.environ["HUGGING_FACE_HUB_TOKEN"] = tok
            print("HF_TOKEN      : loaded from Colab Secrets")
    except Exception:
        print("HF_TOKEN      : not set (OK for Qwen3 / gpt-oss-20b / Goedel-Prover; required for Gemma 4)")

pkgs = []
if importlib.util.find_spec("openai") is None:
    pkgs.append("openai>=1.0")
if importlib.util.find_spec("mlflow") is None:
    pkgs.append("mlflow>=2.0")
if importlib.util.find_spec("vllm") is None:
    pkgs.append("vllm>=0.10.1")
spec = MODEL_REGISTRY[MODEL_NAME]
flags = vllm_flags_for_gpu(spec, compute_cap=compute_cap, vram_gb=vram_gb)
if "bitsandbytes" in flags:
    if importlib.util.find_spec("bitsandbytes") is None:
        pkgs.append("bitsandbytes>=0.46.1")
    if importlib.util.find_spec("vllm_bnb_plugin") is None:
        pkgs.append("vllm-bnb-plugin")
if pkgs:
    print("Installing    :", ", ".join(pkgs))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *pkgs])

prev = globals().get("vllm_proc")
if prev is not None and prev.poll() is None:
    print(f"Stopping previous vLLM process (PID {prev.pid})…")
    prev.terminate()
    try:
        prev.wait(timeout=20)
    except subprocess.TimeoutExpired:
        prev.kill()
        prev.wait()
    time.sleep(2)

VLLM_PORT = 8000
VLLM_URL  = f"http://localhost:{VLLM_PORT}/v1"
VLLM_LOG  = pathlib.Path("/tmp/vllm.log")

cmd = [
    sys.executable, "-m", "vllm.entrypoints.openai.api_server",
    "--model", spec.hf_id,
    "--port", str(VLLM_PORT),
    *flags,
]
print("Launching     :", " ".join(cmd))
vllm_log_fh = open(VLLM_LOG, "wb")
vllm_proc = subprocess.Popen(cmd, stdout=vllm_log_fh, stderr=subprocess.STDOUT)
print(f"vLLM PID      : {vllm_proc.pid}  (waiting for {VLLM_URL}/models…)")

STARTUP_TIMEOUT_S = 600
t0 = time.monotonic()
ready_models = []
while time.monotonic() - t0 < STARTUP_TIMEOUT_S:
    rc = vllm_proc.poll()
    if rc is not None:
        vllm_log_fh.flush()
        tail = VLLM_LOG.read_text(errors="replace").splitlines()[-50:]
        raise RuntimeError(
            f"vLLM exited early (rc={rc}). Last 50 lines of {VLLM_LOG}:\n"
            + "\n".join(tail)
        )
    try:
        with urllib.request.urlopen(f"{VLLM_URL}/models", timeout=3) as resp:
            payload = json.loads(resp.read().decode("utf-8"))
            ready_models = [m.get("id") for m in payload.get("data", [])]
            if spec.hf_id in ready_models:
                break
    except Exception:
        pass
    elapsed = int(time.monotonic() - t0)
    if elapsed > 0 and elapsed % 30 == 0:
        print(f"  still loading {spec.hf_id}… ({elapsed}s)")
    time.sleep(5)
else:
    tail = VLLM_LOG.read_text(errors="replace").splitlines()[-40:]
    raise TimeoutError(
        f"vLLM did not become ready within {STARTUP_TIMEOUT_S}s. Tail of {VLLM_LOG}:\n"
        + "\n".join(tail)
    )

def resolve_hf_commit(hf_id: str) -> str | None:
    cache_dir = pathlib.Path(os.environ.get("HF_HOME", pathlib.Path.home() / ".cache" / "huggingface"))
    repo_dir = cache_dir / "hub" / f"models--{hf_id.replace('/', '--')}"
    ref_main = repo_dir / "refs" / "main"
    if ref_main.is_file():
        return ref_main.read_text().strip()
    snaps = sorted((repo_dir / "snapshots").iterdir()) if (repo_dir / "snapshots").is_dir() else []
    return snaps[-1].name if snaps else None

RESOLVED_HF_COMMIT = spec.revision or resolve_hf_commit(spec.hf_id)
print(f"vLLM READY    : {ready_models}  ({time.monotonic() - t0:.0f}s)")
print(f"HF commit SHA : {RESOLVED_HF_COMMIT or '(not found in local HF cache)'}")

In [ ]:
# ── Cell 4b-run: execute a 200-episode raw LLM cortex bake-off ───────────────
import pathlib, shutil
from prometheus.bakeoff import BakeoffConfig, BakeoffRunner, MODEL_REGISTRY
from prometheus.runtime.gate import Halted

MODEL_NAME       = globals().get("MODEL_NAME", "qwen3-8b")
spec             = MODEL_REGISTRY[MODEL_NAME]
LOCAL_MLFLOW_DB  = pathlib.Path("/content/mlflow.db")
DRIVE_MLFLOW_DB  = pathlib.Path(f"{PROMETHEUS_DIR}/mlflow.db")
CKPT_PATH        = f"{PROMETHEUS_DIR}/bakeoff_{MODEL_NAME}.json"
VLLM_URL         = globals().get("VLLM_URL", "http://localhost:8000/v1")

if DRIVE_MLFLOW_DB.exists() and not LOCAL_MLFLOW_DB.exists():
    shutil.copy2(DRIVE_MLFLOW_DB, LOCAL_MLFLOW_DB)

def snapshot_mlflow_db() -> None:
    if LOCAL_MLFLOW_DB.exists():
        tmp = DRIVE_MLFLOW_DB.with_name(f".{DRIVE_MLFLOW_DB.name}.tmp")
        shutil.copy2(LOCAL_MLFLOW_DB, tmp)
        tmp.replace(DRIVE_MLFLOW_DB)

gate.check_heartbeat()

MLFLOW_URI = f"sqlite:///{LOCAL_MLFLOW_DB}"
print(f"Model      : {MODEL_NAME}  ({spec.hf_id} @ {spec.revision or 'HEAD'})")
print(f"Max tokens : {spec.default_max_tokens}")
print(f"MLflow URI : {MLFLOW_URI}  (snapshotted to {DRIVE_MLFLOW_DB})")
print(f"Checkpoint : {CKPT_PATH}")
print()

config = BakeoffConfig(
    model_name=MODEL_NAME,
    n_episodes=200,
    noise=0.05,
    budget=500.0,
    vllm_base_url=VLLM_URL,
    temperature=0.0,
    max_tokens=spec.default_max_tokens,
    mlflow_tracking_uri=MLFLOW_URI,
    mlflow_experiment="phase0-bakeoff",
    checkpoint_path=CKPT_PATH,
    check_health=True,
)

runner = BakeoffRunner(config)

def _on_episode(episode_idx, result):
    try:
        gate.check_heartbeat()
    except Halted as exc:
        snapshot_mlflow_db()
        print(f"\nBake-off halted by governor at episode {episode_idx + 1}: {exc}")
        raise SystemExit(0)
    if episode_idx == 0 or (episode_idx + 1) % 20 == 0 or result.error:
        snapshot_mlflow_db()
        tag = f"[ERROR: {result.error[:60]}]" if result.error else ""
        print(f"  Episode {episode_idx+1:4d}/200  "
              f"family={result.family:20s}  acc={result.accuracy:.3f}  "
              f"lat={result.latency_s:.1f}s  {tag}")

try:
    summary = runner.run(on_episode=_on_episode)
finally:
    snapshot_mlflow_db()

print()
print(f"═══ Bake-off complete: {MODEL_NAME} ═══")
print(f"HF commit SHA  : {globals().get('RESOLVED_HF_COMMIT') or spec.revision or '(run Cell 4b-setup)'}")
print(f"Accuracy       : {summary.accuracy:.3f}  ({summary.n_errors} errors)")
print(f"Mean latency   : {summary.mean_latency_s:.2f}s / episode")

In [ ]:
# ── Cell 4b-compare: leaderboard across raw bake-offs + Phase 1 & Phase 2 ───
import json, pathlib
from prometheus.bakeoff import MODEL_REGISTRY
from prometheus.lab.laws import LAW_NAMES

CANDIDATES = [k for k in MODEL_REGISTRY if k != "goedel-prover-v2-8b"]
rows = []

for name in CANDIDATES:
    ckpt = pathlib.Path(f"{PROMETHEUS_DIR}/bakeoff_{name}.json")
    if not ckpt.exists():
        continue
    state = json.loads(ckpt.read_text())
    results = list(state["results"].values())
    if not results:
        continue
    n = len(results)
    total_correct = sum(
        0 if r.get("error") else sum(r["claims"].get(k) == r["truth"].get(k) for k in LAW_NAMES)
        for r in results
    )
    acc = total_correct / (n * len(LAW_NAMES))
    n_errors = sum(1 for r in results if r.get("error"))
    lats = [r.get("latency_s", 0) for r in results]
    mean_lat = sum(lats) / len(lats) if lats else 0
    rows.append((name, n, acc, n_errors, mean_lat))

for label, pth in [("phase1-cognitive-core", PHASE1_CKPT_PATH), ("phase2-transfer-ashby", PHASE2_CKPT_PATH)]:
    p = pathlib.Path(pth)
    if p.exists():
        st = json.loads(p.read_text())
        res = list(st.get("results", {}).values())
        if res:
            n = len(res)
            acc = sum(r["correct"] for r in res) / (n * len(LAW_NAMES))
            mean_lat = sum(r.get("latency_s", 0) for r in res) / n
            rows.append((label, n, acc, 0, mean_lat))

if rows:
    rows.sort(key=lambda r: -r[2])
    print(f"{'Model / Agent':<24} {'N':>5} {'Accuracy':>9} {'Errors':>7} {'Lat(s)':>8}")
    print("─" * 58)
    for name, n, acc, nerr, lat in rows:
        winner = " ◀ BEST" if rows and rows[0][0] == name and len(rows) > 1 else ""
        print(f"{name:<24} {n:>5} {acc:>9.3f} {nerr:>7} {lat:>8.2f}{winner}")
else:
    print("No checkpoints found yet.")

---
## Cell 4c — Audit-seed Evaluation *(read-only memory, no ground truth)*

Evaluates `AUDIT_SEEDS` (`1000–1049`) using `CognitiveAgent` + `StructuralAnalogyTransfer`
in read-only memory mode (`learn=False`) and checkpoints claims to `audit_checkpoint.json`.

In [ ]:
import hashlib, json, pathlib
from prometheus.core import (
    AssemblyMemory,
    CognitiveAgent,
    StructuralAnalogyTransfer,
    load_harness,
)
from prometheus.lab import HiddenLawWorld
from prometheus.lab.laws import LAW_NAMES
from prometheus.runtime.gate import Halted

AUDIT_NOISE  = 0.05
AUDIT_BUDGET = 500.0

harness_dir = pathlib.Path(INSTALL_ROOT) / "current" / "harness"
harness = load_harness(harness_dir if harness_dir.is_dir() else None)
memory = AssemblyMemory.load(pathlib.Path(MEMORY_STATE_PATH), params_override=harness.memory_params)
pre_audit_eps = memory.episodes_consolidated

transfer = StructuralAnalogyTransfer(harness.transfer_params)
agent = CognitiveAgent(harness=harness, memory=memory, transfer=transfer, rng_seed=1000)

sig_payload = json.dumps({
    "mode": "phase2_cognitive_agent",
    "efe": harness.efe_params.__dict__,
    "mem_params": harness.memory_params.__dict__,
    "transfer_params": harness.transfer_params.__dict__,
    "mesh": harness.mesh.__dict__,
    "assemblies": len(memory.assemblies),
    "episodes_consolidated": memory.episodes_consolidated,
}, sort_keys=True, default=str)
eval_hash = hashlib.sha256(sig_payload.encode("utf-8")).hexdigest()[:16]

audit_ckpt_path = pathlib.Path(
    globals().get("AUDIT_CHECKPOINT_PATH", f"{EXCHANGE_DIR}/audit_checkpoint.json")
)
if audit_ckpt_path.exists():
    audit_state = json.loads(audit_ckpt_path.read_text(encoding="utf-8"))
    if audit_state.get("code_hash") != eval_hash:
        print("Harness/memory changed — starting fresh audit_checkpoint.json.")
        audit_state = {"code_hash": eval_hash, "results": {}}
    else:
        print(f"Resuming audit_checkpoint.json: {len(audit_state['results'])}/{len(AUDIT_SEEDS)} done")
else:
    audit_state = {"code_hash": eval_hash, "results": {}}

def _save_audit_checkpoint(st: dict) -> None:
    tmp = audit_ckpt_path.with_name(f".{audit_ckpt_path.name}.tmp")
    tmp.write_text(json.dumps(st, indent=2, sort_keys=True), encoding="utf-8")
    tmp.replace(audit_ckpt_path)

for idx, seed in enumerate(AUDIT_SEEDS, start=1):
    key = str(seed)
    if key in audit_state["results"]:
        continue

    try:
        gate.check_heartbeat()
    except Halted as e:
        _save_audit_checkpoint(audit_state)
        print(f"\nAudit evaluation halted by governor at seed {seed}: {e}")
        raise SystemExit(0)

    world = HiddenLawWorld(seed=seed, noise=AUDIT_NOISE, budget=AUDIT_BUDGET)
    res = agent.run_episode(world, learn=False, episode_idx=idx)
    audit_state["results"][key] = {
        "claims": {k: bool(res.claims[k]) for k in LAW_NAMES},
        "n": res.n,
        "spent": round(res.budget_spent, 3),
        "observations": res.observations_used,
        "entropy_bits": round(res.final_entropy_bits, 4),
        "transfer_hints": res.transfer_hints,
    }
    _save_audit_checkpoint(audit_state)

    if idx % 10 == 0 or idx == len(AUDIT_SEEDS):
        print(f"Audit seeds evaluated: {len(audit_state['results'])}/{len(AUDIT_SEEDS)} "
              f"(last budget={res.budget_spent:.1f}, entropy={res.final_entropy_bits:.3f} bits)")

assert memory.episodes_consolidated == pre_audit_eps, "audit must never mutate memory!"
mean_spent = sum(r["spent"] for r in audit_state["results"].values()) / len(audit_state["results"])
print(f"Saved {len(audit_state['results'])}/{len(AUDIT_SEEDS)} audit claims "
      f"to {audit_ckpt_path.name} (mean budget={mean_spent:.1f})")

---
## Cell 4d — Lean 4 Prover Bake-off *(optional / Goedel-Prover-V2-8B)*

Installs Lean 4 via `elan` eagerly (no 15s timeout), verifies `seed=0`, and benchmarks
whichever model **Cell 4b-setup** is serving on `localhost:8000`.

In [ ]:
# ── Cell 4d-setup: install Lean 4 (elan) on Colab & sanity-check ─────────────
import os, pathlib, subprocess
from prometheus.lab import HiddenLawWorld
from prometheus.lab.lean_layer import lean_version, verify_world

LEAN_TOOLCHAIN = "leanprover/lean4:v4.34.1"

elan_bin = pathlib.Path.home() / ".elan" / "bin"
if str(elan_bin) not in os.environ.get("PATH", ""):
    os.environ["PATH"] = f"{elan_bin}:{os.environ.get('PATH', '')}"

if not (elan_bin / "elan").exists():
    print("Installing elan…")
    subprocess.check_call([
        "bash", "-c",
        "curl -sSf https://raw.githubusercontent.com/leanprover/elan/master/elan-init.sh "
        "| sh -s -- -y --default-toolchain none",
    ])

print(f"Ensuring toolchain {LEAN_TOOLCHAIN} is installed…")
subprocess.check_call([str(elan_bin / "elan"), "toolchain", "install", LEAN_TOOLCHAIN])
subprocess.check_call([str(elan_bin / "elan"), "default", LEAN_TOOLCHAIN])

print("Lean version  :", lean_version())
ok, err = verify_world(HiddenLawWorld(seed=0), timeout=60)
assert ok, f"Lean sanity check failed: {err}"
print("Sanity check  : seed=0 ground-truth proof verified OK!")

In [ ]:
# ── Cell 4d-run: resumable Lean 4 prover benchmark ───────────────────────────
import json, pathlib
from collections import defaultdict
from prometheus.bakeoff import MODEL_REGISTRY
from prometheus.harness.client import ModelClient, ModelClientError
from prometheus.harness.prover import run_prover_episode
from prometheus.lab import HiddenLawWorld
from prometheus.runtime.gate import Halted

PROVER_MODEL_NAME = globals().get("MODEL_NAME", "goedel-prover-v2-8b")
PROVER_SEEDS      = list(range(50))
PROVER_CKPT_PATH  = pathlib.Path(f"{PROMETHEUS_DIR}/prover_{PROVER_MODEL_NAME}.json")
VLLM_URL          = globals().get("VLLM_URL", "http://localhost:8000/v1")

spec = MODEL_REGISTRY[PROVER_MODEL_NAME]
client = ModelClient(
    base_url=VLLM_URL,
    model_id=spec.hf_id,
    temperature=0.0,
    max_tokens=max(spec.default_max_tokens, 4096),
    extra_body=dict(spec.extra_body),
)

gate.check_heartbeat()
if not client.health_check():
    raise ModelClientError(
        f"vLLM at {VLLM_URL} is not serving {spec.hf_id!r}. "
        f"Set MODEL_NAME = {PROVER_MODEL_NAME!r} in Cell 4b-setup and run it first."
    )

if PROVER_CKPT_PATH.exists():
    pstate = json.loads(PROVER_CKPT_PATH.read_text(encoding="utf-8"))
    print(f"Resuming {PROVER_CKPT_PATH.name}: {len(pstate['results'])}/{len(PROVER_SEEDS)} done")
else:
    pstate = {
        "model_name": PROVER_MODEL_NAME,
        "hf_id": spec.hf_id,
        "hf_commit": globals().get("RESOLVED_HF_COMMIT") or spec.revision,
        "lean_toolchain": globals().get("LEAN_TOOLCHAIN", "unknown"),
        "results": {},
    }

def _save_prover_ckpt(st: dict) -> None:
    tmp = PROVER_CKPT_PATH.with_name(f".{PROVER_CKPT_PATH.name}.tmp")
    tmp.write_text(json.dumps(st, indent=2, sort_keys=True), encoding="utf-8")
    tmp.replace(PROVER_CKPT_PATH)

for idx, seed in enumerate(PROVER_SEEDS, start=1):
    key = str(seed)
    if key in pstate["results"]:
        continue

    try:
        gate.check_heartbeat()
    except Halted as e:
        _save_prover_ckpt(pstate)
        print(f"\nProver bake-off halted by governor at seed {seed}: {e}")
        raise SystemExit(0)

    world = HiddenLawWorld(seed=seed)
    res = run_prover_episode(world, client, lean_timeout=60)
    pstate["results"][key] = res.to_dict()
    _save_prover_ckpt(pstate)

    if idx % 5 == 0 or not res.proved or idx == len(PROVER_SEEDS):
        done = len(pstate["results"])
        n_ok = sum(1 for r in pstate["results"].values() if r["proved"])
        tag = "PASS" if res.proved else f"FAIL ({(res.error or '')[:70]})"
        print(f"  Seed {seed:3d} ({done:2d}/{len(PROVER_SEEDS)})  "
              f"family={res.family:19s}  lat={res.latency_s:4.1f}s  "
              f"pass={n_ok}/{done}  {tag}")

rows = list(pstate["results"].values())
n_total = len(rows)
n_proved = sum(1 for r in rows if r["proved"])
mean_lat = sum(r["latency_s"] for r in rows) / n_total if n_total else 0.0
print(f"\nProver {PROVER_MODEL_NAME}: {n_proved}/{n_total} ({n_proved/n_total:.3f}), mean_lat={mean_lat:.2f}s")

---
## Cell 4e — Phase 2 Bounded Ashby Self-Improvement Loop *(autonomous `g3` proposal)*

Demonstrates the full **Ashby + Hofstadter + Governor** self-improvement loop:
1. Checks `HomeostaticMonitor` (or runs a tight-budget `BUDGET=22.0` stress probe on
   held-out dev seeds `300–311`, strictly disjoint from `AUDIT_SEEDS` `1000–1049`).
2. `AshbyProposer` generates validated candidate mutations under `harness/memory_params/`
   and `harness/mesh/` (optionally consulting live Qwen3-8B if vLLM is running).
3. Screens every candidate on `dev_seeds` (read-only memory copy), audits the winner on
   `AUDIT_SEEDS` (`learn=False`, no ground truth!), and publishes `g3` + `.ready` to Drive.
4. Run **Cell 6** right after to wait for the workstation governor's signed decision
   and install generation 3!

In [ ]:
import pathlib
from prometheus.bakeoff import MODEL_REGISTRY
from prometheus.common.eventlog import EventLog
from prometheus.core import (
    AshbyProposer,
    AssemblyMemory,
    CognitiveAgent,
    HomeostaticMonitor,
    StressDiagnosis,
    StructuralAnalogyTransfer,
    load_harness,
)
from prometheus.harness.client import ModelClient
from prometheus.lab import HiddenLawWorld

ASHBY_VARIANT_NAME  = "g3"                  # generation 3 (g2 is installed at gen 2)
DEV_SEEDS           = list(range(300, 312)) # strictly disjoint from AUDIT_SEEDS (1000-1049)
DEV_BUDGET          = 25.0                  # tight budget highlights transfer + efficiency gains
USE_LLM_PROPOSER    = False                 # set True if vLLM is running in Cell 4b-setup
# Keep False on run-001 where g2 already saturated audit_score=1.000 (set True to test proxy-gain veto!)
INCLUDE_PROXY_SCORE = False

existing_decision = pathlib.Path(EXCHANGE_DIR) / "approvals" / f"{ASHBY_VARIANT_NAME}.json"
if existing_decision.exists():
    raise RuntimeError(
        f"{existing_decision.name} already exists — pick a fresh ASHBY_VARIANT_NAME (e.g. '{ASHBY_VARIANT_NAME}b')."
    )

gate.check_heartbeat()

harness_dir = pathlib.Path(INSTALL_ROOT) / "current" / "harness"
current_harness = load_harness(harness_dir if harness_dir.is_dir() else None)
memory = AssemblyMemory.load(pathlib.Path(MEMORY_STATE_PATH), params_override=current_harness.memory_params)
agent_log = EventLog(pathlib.Path(AGENT_LOG_PATH))
monitor = HomeostaticMonitor.load(
    pathlib.Path(ASHBY_STATE_PATH), config_override=current_harness.ashby_config, event_log=agent_log
)

# If monitor is currently homeostatic, run a tight-budget stress probe to obtain a real StressDiagnosis
diagnosis = monitor.last_diagnosis
if not diagnosis.reasons:
    print(f"Monitor currently HOMEOSTATIC — running tight-budget stress probe (budget={DEV_BUDGET})…")
    probe_agent = CognitiveAgent(
        harness=current_harness,
        memory=memory,
        transfer=StructuralAnalogyTransfer(current_harness.transfer_params),
        monitor=monitor,
        rng_seed=300,
    )
    for idx, s in enumerate(DEV_SEEDS[:6]):
        gate.check_heartbeat()
        probe_agent.run_episode(HiddenLawWorld(seed=s, noise=0.05, budget=DEV_BUDGET), learn=False, episode_idx=idx)
    monitor.save(pathlib.Path(ASHBY_STATE_PATH))
    diagnosis = monitor.last_diagnosis
    if not diagnosis.reasons:
        diagnosis = StressDiagnosis(
            triggered=True,
            reasons=("HIGH_BUDGET",),
            violations=("scheduled tight-budget optimisation round",),
            rolling_metrics=monitor.status_summary().get("rolling_metrics", {}),
        )

print(f"Stress diagnosis : reasons={diagnosis.reasons}")

llm_client = None
if USE_LLM_PROPOSER:
    spec = MODEL_REGISTRY[globals().get("MODEL_NAME", "qwen3-8b")]
    cand_client = ModelClient(
        base_url=globals().get("VLLM_URL", "http://localhost:8000/v1"),
        model_id=spec.hf_id,
        temperature=0.2,
        max_tokens=1024,
        extra_body=dict(spec.extra_body),
    )
    if cand_client.health_check():
        llm_client = cand_client
        print(f"LLM proposer     : ON ({spec.hf_id})")

proposer = AshbyProposer(client=llm_client, event_log=agent_log, max_generations=10)
print(f"Screening Ashby candidates on {len(DEV_SEEDS)} dev worlds, then auditing {len(AUDIT_SEEDS)} audit worlds…")
outcome, evals = proposer.screen_and_publish(
    variant_name=ASHBY_VARIANT_NAME,
    current_harness=current_harness,
    diagnosis=diagnosis,
    memory=memory,
    gate=gate,
    exchange=pathlib.Path(EXCHANGE_DIR),
    dev_seeds=DEV_SEEDS,
    audit_seeds=AUDIT_SEEDS,
    model_pins=MODEL_PINS,
    noise=0.05,
    budget=DEV_BUDGET,
    audit_budget=500.0,
    use_transfer=True,
    include_proxy_score=INCLUDE_PROXY_SCORE,
    rng_seed=300,
)

print()
print(f"{'Candidate':<22} {'DevAcc':>8} {'Budget':>8} {'Conf':>7} {'Viols':>6} {'Proxy':>7}")
print("─" * 62)
for ev in evals:
    star = " ◀ SELECTED" if ev.label == outcome.candidate_label else ""
    print(f"{ev.label:<22} {ev.dev_accuracy:>8.4f} {ev.mean_budget_spent:>8.2f} "
          f"{ev.mean_confidence:>7.3f} {ev.bound_violations:>6d} {ev.proxy_score:>7.4f}{star}")

print()
print(f"Published     : {outcome.variant_name} (generation={outcome.generation})")
print(f"Variant hash  : {outcome.variant_hash}")
print(f"Files         : {list(outcome.files)}")
print("Next step     : run Cell 6 (VARIANT_TO_INSTALL = 'g3') to wait for Governor approval & install.")

---
## Cell 5 — Manual Propose a Harness Variant *(optional if you didn't use Cell 4e)*
Packages validated Phase 1/2 mutable harness files (`harness/memory_params/`,
`harness/mesh/`, `harness/prompts/`) with `audit_claims.json` from **Cell 4c**
and writes `.ready`.

In [ ]:
import dataclasses, json, pathlib, shutil, time
from prometheus.common.variant import write_bundle
from prometheus.core import AshbyConfig, EFEParams, MemoryParams, MeshConfig, TransferParams

VARIANT_NAME   = "g3"
HARNESS_DIR    = pathlib.Path(f"{INSTALL_ROOT}/current/harness")
PROXY_SCORE    = None
AUDIT_CKPT     = pathlib.Path(
    globals().get("AUDIT_CHECKPOINT_PATH", f"{EXCHANGE_DIR}/audit_checkpoint.json")
)

existing_decision = pathlib.Path(EXCHANGE_DIR) / "approvals" / f"{VARIANT_NAME}.json"
if existing_decision.exists():
    raise RuntimeError(
        f"{existing_decision.name} already exists — pick a new VARIANT_NAME (e.g. '{VARIANT_NAME}b')."
    )

if not AUDIT_CKPT.exists():
    raise RuntimeError(f"{AUDIT_CKPT.name} not found — run Cell 4c first.")
cp_state = json.loads(AUDIT_CKPT.read_text(encoding="utf-8"))
audit_claims = {
    str(seed): {k: bool(v) for k, v in cp_state["results"][str(seed)]["claims"].items()}
    for seed in AUDIT_SEEDS
    if str(seed) in cp_state.get("results", {})
}
if len(audit_claims) < len(AUDIT_SEEDS):
    raise RuntimeError(
        f"Only {len(audit_claims)}/{len(AUDIT_SEEDS)} AUDIT_SEEDS found in {AUDIT_CKPT.name}."
    )

harness_files: dict[str, str] = {}
if HARNESS_DIR.exists():
    for p in sorted(HARNESS_DIR.rglob("*")):
        if p.is_file() and not p.is_symlink():
            rel = f"harness/{p.relative_to(HARNESS_DIR).as_posix()}"
            harness_files[rel] = p.read_text(encoding="utf-8")

harness_files.setdefault(
    "harness/memory_params/efe_params.json",
    json.dumps(dataclasses.asdict(EFEParams()), indent=2, sort_keys=True) + "\n",
)
harness_files.setdefault(
    "harness/memory_params/memory_params.json",
    json.dumps(dataclasses.asdict(MemoryParams()), indent=2, sort_keys=True) + "\n",
)
harness_files.setdefault(
    "harness/memory_params/transfer_params.json",
    json.dumps(dataclasses.asdict(TransferParams()), indent=2, sort_keys=True) + "\n",
)
harness_files.setdefault(
    "harness/mesh/config.json",
    json.dumps(dataclasses.asdict(MeshConfig()), indent=2, sort_keys=True) + "\n",
)
harness_files.setdefault(
    "harness/mesh/ashby_params.json",
    json.dumps(dataclasses.asdict(AshbyConfig()), indent=2, sort_keys=True) + "\n",
)

proposals_root = pathlib.Path(EXCHANGE_DIR) / "proposals"
proposals_root.mkdir(parents=True, exist_ok=True)
staging_dir  = proposals_root / f".staging-{VARIANT_NAME}"
proposal_dir = proposals_root / VARIANT_NAME
if staging_dir.exists():
    shutil.rmtree(staging_dir)
if proposal_dir.exists():
    shutil.rmtree(proposal_dir)

manifest_fields = {
    "run_id":      RUN_ID,
    "generation":  gate.generation + 1,
    "parent_hash": gate.installed_hash,
    "model_pins":  MODEL_PINS,
    "created_at":  time.time(),
}
if PROXY_SCORE is not None:
    manifest_fields["proxy_score"] = PROXY_SCORE

bundle = write_bundle(staging_dir, manifest_fields, harness_files)
(staging_dir / "audit_claims.json").write_text(
    json.dumps(audit_claims, indent=2, sort_keys=True), encoding="utf-8"
)
staging_dir.rename(proposal_dir)
(proposal_dir / ".ready").write_text(bundle.variant_hash + "\n", encoding="utf-8")

print(f"Bundle name   : {VARIANT_NAME}")
print(f"Variant hash  : {bundle.variant_hash}")
print(f"Generation    : {manifest_fields['generation']}")
print(f"Files         : {sorted(harness_files)}")
print("Bundle published. Run Cell 6 to wait for Governor approval & install.")

---
## Cell 6 — Wait for approval & install variant (`g3`)
Polls Drive until `exchange/approvals/<name>.json` arrives (up to one 5-minute
governor tick plus rclone sync), verifies the signed token via `gate.install()`,
atomically swaps the harness onto `/content/installed_harness`, and verifies
`load_harness()` picks up all Phase 2 configs (`transfer_params.json`, `ashby_params.json`).

In [ ]:
import pathlib, time
from prometheus.core import load_harness
from prometheus.runtime.gate import Halted, InstallRefused

VARIANT_TO_INSTALL = "g3"    # matches Cell 4e's ASHBY_VARIANT_NAME (or Cell 5's VARIANT_NAME)
WAIT_FOR_APPROVAL  = True
WAIT_TIMEOUT_S     = 420     # 7 min (> 5-min governor tick + rclone sync)
POLL_INTERVAL_S    = 15

proposal_dir  = pathlib.Path(EXCHANGE_DIR) / "proposals" / VARIANT_TO_INSTALL
approval_path = pathlib.Path(EXCHANGE_DIR) / "approvals" / f"{VARIANT_TO_INSTALL}.json"

if WAIT_FOR_APPROVAL and not approval_path.is_file():
    start = time.monotonic()
    deadline = start + WAIT_TIMEOUT_S
    print(
        f"Waiting for {approval_path.name} (up to {WAIT_TIMEOUT_S}s; "
        "governor ticks every 5 min)…"
    )
    while not approval_path.is_file():
        gate.check_heartbeat()
        remaining = deadline - time.monotonic()
        if remaining <= 0:
            print(f"Timed out after {WAIT_TIMEOUT_S}s waiting for {approval_path.name}.")
            break
        time.sleep(min(POLL_INTERVAL_S, remaining))
    if approval_path.is_file():
        print(f"Decision arrived: {approval_path.name} ({time.monotonic() - start:.0f}s)")

try:
    dest = gate.install(proposal_dir)
    installed_files = sorted(
        p.relative_to(dest).as_posix() for p in dest.rglob("*") if p.is_file()
    )
    loaded = load_harness(pathlib.Path(INSTALL_ROOT) / "current" / "harness")
    print(f"Installed     : generation={gate.generation}, dest={dest}")
    print(f"Installed hash: {gate.installed_hash}")
    print(f"Files         : {installed_files}")
    print(f"Harness reload: OK ({loaded.loaded_files})")
except InstallRefused as e:
    print(f"Install refused: {e}")
    raise
except Halted as e:
    print(f"Halted before install: {e}")
    raise